# Weigh a colloid by counting where it spends its time

Brownian motion spreads a dilute suspension through a cell whilst buoyancy biases its vertical position. Derive the equilibrium concentration, predict counts in finite camera bins and estimate a buoyant mass without measuring how quickly any individual sphere falls.

## Balance drift against diffusion

Measure $z$ upwards from the bottom of a closed cell. A sphere of volume $V$ displaces fluid of density $\rho_f$. The downward gravitational force on the particle exceeds the upward buoyancy force by $m_bg$, where $m_b=V(\rho_p-\rho_f)$. Signed buoyant mass is what a concentration profile can reveal. Let $\mu$ denote mobility, the ratio of speed to applied force. For dilute, independently moving particles at uniform temperature, the total vertical number flux is

$$J=-D\frac{dn}{dz}-\mu m_b g n. $$

At steady state the flux is constant, and impermeable cell ends set that constant to zero. The Einstein relation $D=\mu k_BT$ eliminates the mobility. Integrating $d\ln n/dz=-a$ with $a=m_bg/(k_BT)$ gives

$$
n(z)=n(0)e^{-az}.
$$

The signed slope changes sign for buoyant particles and vanishes at exact density matching. Viscosity affects the time to reach equilibrium.

Suppose the camera counts centres in equal-width horizontal slices of a column with uniform cross-section. A bin centred at $z_i$ has expected count proportional to $e^{-az_i} \int_{-h/2}^{h/2}e^{-au}du$. The integral is common to all bins, hence normalising the exponential values at their centres yields the exact bin probabilities for this model. Wall exclusion sets the interval accessible to centres. Define the specified cell height as that accessible interval.

## Infer a mass with its sign

Use a radius of 0.50 µm, particle excess density 50 kg/m³ and temperature 293 K. The volume is $5.236\times10^{-19}$ m³, the buoyant mass is $2.618\times10^{-17}$ kg and $k_BT/(m_bg)=15.75$ µm. Counts in two equal slices separated by 10 µm should have lower-to-upper ratio $e^{10/15.75}\simeq1.89$. Doubling the radius at fixed density ratio multiplies the mass by eight and divides the gravitational length by eight.

Sampling is important when using the upper part of a steep profile. Suppose bins separated by 10 µm contain 120 and 60 approximately independent particle observations. The estimated slope is $\ln(2)/10=0.0693$ µm⁻¹. For independent Poisson counts the standard error from propagating the two count variances is $\sqrt{1/120+1/60}/10=0.0158$ µm⁻¹. Repeatedly imaging the same particle at closely spaced times introduces correlations. Counting all those frames as independent would make the quoted sampling precision too optimistic.

## Set the experimental inputs

Edit these values, then run the calculation and both figures again.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

teal, orange, blue = "#19596a", "#aa462a", "#3868a0"
def axes(xlabel, ylabel):
    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    ax.set(xlabel=xlabel, ylabel=ylabel)
    ax.grid(alpha=0.2)
    return fig, ax

In [ ]:
radius_um = .50
excess_density_kg_m3 = 50.
temperature_K = 293.
accessible_height_um = 60.
bins = 12
observations = 1200
seed = 22

In [ ]:
assert radius_um>0 and temperature_K>0 and accessible_height_um>0
assert isinstance(bins,int) and bins>=2 and isinstance(observations,int) and observations>0
boltzmann = 1.380649e-23
volume = 4*np.pi*(radius_um*1e-6)**3/3
a = volume*excess_density_kg_m3*9.80665/(boltzmann*temperature_K)*1e-6
width = accessible_height_um/bins
centres = (np.arange(bins)+.5)*width
def bin_probability(slope):
    log_weights = -slope*centres
    weights = np.exp(log_weights-log_weights.max())
    return weights / weights.sum()

probability = bin_probability(a)
counts = np.random.default_rng(seed).multinomial(observations,probability)
assert np.allclose(bin_probability(-a),probability[::-1])
assert counts.sum()==observations
print(f"Buoyant mass {volume*excess_density_kg_m3:.4e} kg; signed slope {a:.5f} per µm")

In [ ]:
fig,ax = axes("Height of slice centre (µm)", "Probability in a slice")
for slope, label, color in [(abs(a),"Denser spheres",teal),(0,"Density matched","grey"),(-abs(a),"Lighter spheres",orange)]:
    ax.plot(centres,bin_probability(slope),".-",color=color,label=label)
ax.legend()
plt.show()

The total probability in the cell is one for each curve. Matching the fluid density removes the gravity-induced concentration gradient.

In [ ]:
fig,ax = axes("Height of slice centre (µm)", "Number of independent observations")
ax.bar(centres,counts,width=.8*width,alpha=.6,color=teal,label="Sample counts")
ax.plot(centres,observations*probability,"o-",color=orange,label="Exact expectation")
ax.legend()
plt.show()

Reduce the number of independent particle observations and change the random seed. Noise is particularly large relative to the expected number in sparsely occupied slices.

## Exercise · determine the size of a density-matched particle

A uniform profile persists in a well-equilibrated, isothermal cell. Can gravity alone distinguish two different radii if the particles have the same density as the fluid?

At exact density matching, $m_b=V(\rho_p-\rho_f)=0$ for either radius, so both profiles are uniform. Their diffusion times can still differ because hydrodynamic mobility depends on size. A time-resolved measurement carries information absent from the equilibrium profile.

## Exercise · reverse the measured gradient

Two equally wide slices separated vertically by 12 µm contain 80 and 160 observations, in order of increasing height. Estimate $m_b$ at 293 K.

The signed slope is $\ln(80/160)/(12\ \mathrm{µm})=-0.05776$ µm⁻¹. Converting to metres and multiplying by $k_BT/g$ gives $m_b\simeq-2.38\times10^{-17}$ kg. The sign indicates that the displaced fluid is heavier than the particle.